# EDA — Anomaly / OOD Detection Results

Exploratory analysis of the anomaly-detection sweep across the four EoMT/ERFNet checkpoints, five OOD datasets, and four scoring methods.

**Input:** `all_results_by_checkpoint.json` (produced by `collect_results_by_checkpoint.py`, which separates ERFNet / EoMT_CS / EoMT_COCO / EoMT_FT so the three EoMT checkpoints no longer overwrite each other).

**Outputs (per temperature):**
- `auprc_heatmap_*.png`
- `fpr95_heatmap_*.png`
- `auprc_vs_fpr95_*.png`
- `best_method_per_dataset_*.png`
- `report_table_*.csv`


In [ ]:
# ---- Config: edit these paths, then Run All ----
RESULTS_JSON = "/content/drive/MyDrive/anom_project/results/all_results_by_checkpoint.json"
OUT_DIR      = "/content/drive/MyDrive/anom_project/results/eda"
TEMPERATURE  = "T1"   # which temperature key to read from the results JSON


In [ ]:
import csv
import json
from pathlib import Path
from typing import Dict, List, Any

import matplotlib.pyplot as plt
import numpy as np

DATASET_ORDER = ["RA21", "RO21", "LAF", "fs_static", "RA"]
VARIANT_ORDER = ["ERFNet", "EoMT_CS", "EoMT_COCO", "EoMT_FT", "EoMT_UNKNOWN"]
METHOD_ORDER  = ["msp", "maxlogit", "maxentropy", "rba"]


## Load results

In [ ]:
def load_rows(path: str, temp_key: str = "T1") -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for dataset, variants in data.items():
        for variant, methods in variants.items():
            for method, temps in methods.items():
                if temp_key not in temps:
                    continue
                m = temps[temp_key]
                rows.append({
                    "dataset": dataset,
                    "variant": variant,
                    "method": method,
                    "label": f"{variant}:{method}",
                    "auprc": float(m["auprc"]),
                    "fpr95": float(m["fpr95"]),
                    "images_used": m.get("images_used"),
                })
    return rows


def ordered_unique(values, order):
    return [x for x in order if x in values] + sorted([x for x in values if x not in order])


def pivot_best_by_label(rows: List[Dict[str, Any]], metric: str):
    datasets = ordered_unique({r["dataset"] for r in rows}, DATASET_ORDER)
    labels = sorted({r["label"] for r in rows})
    mat = np.full((len(labels), len(datasets)), np.nan)
    for i, label in enumerate(labels):
        for j, ds in enumerate(datasets):
            vals = [r[metric] for r in rows if r["label"] == label and r["dataset"] == ds]
            if vals:
                mat[i, j] = vals[0]
    return labels, datasets, mat


In [ ]:
out_dir = Path(OUT_DIR).expanduser()
out_dir.mkdir(parents=True, exist_ok=True)

rows = load_rows(RESULTS_JSON, TEMPERATURE)
if not rows:
    raise RuntimeError(f"No rows found for temperature key {TEMPERATURE}")

print(f"Loaded {len(rows)} rows at temperature {TEMPERATURE}")
print("Datasets:", sorted({r['dataset'] for r in rows}))
print("Variants:", sorted({r['variant'] for r in rows}))
print("Methods :", sorted({r['method'] for r in rows}))


## Plotting functions

In [ ]:
def save_heatmap(rows, metric, out_path, title):
    labels, datasets, mat = pivot_best_by_label(rows, metric)
    fig, ax = plt.subplots(figsize=(max(7, len(datasets) * 1.2), max(5, len(labels) * 0.35)))
    im = ax.imshow(mat, aspect="auto")
    ax.set_xticks(range(len(datasets)))
    ax.set_xticklabels(datasets, rotation=30, ha="right")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metric.upper())
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:.1f}", ha="center", va="center", fontsize=7)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    print(f"[SAVED] {out_path}")
    plt.show()


def save_scatter(rows, out_path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for r in rows:
        ax.scatter(r["fpr95"], r["auprc"])
        ax.annotate(f"{r['dataset']}\n{r['label']}", (r["fpr95"], r["auprc"]), fontsize=6, alpha=0.8)
    ax.set_xlabel("FPR95 (%) — lower is better")
    ax.set_ylabel("AuPRC (%) — higher is better")
    ax.set_title("AuPRC vs FPR95 at T=1.0")
    ax.grid(True, linewidth=0.3, alpha=0.5)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    print(f"[SAVED] {out_path}")
    plt.show()


def save_best_bar(rows, out_path):
    datasets = ordered_unique({r["dataset"] for r in rows}, DATASET_ORDER)
    best, labels = [], []
    for ds in datasets:
        ds_rows = [r for r in rows if r["dataset"] == ds]
        if not ds_rows:
            continue
        b = max(ds_rows, key=lambda x: x["auprc"])
        best.append(b["auprc"])
        labels.append(f"{ds}\n{b['label']}")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(range(len(best)), best)
    ax.set_xticks(range(len(best)))
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel("Best AuPRC (%) at T=1.0")
    ax.set_title("Best method/checkpoint per dataset")
    for i, v in enumerate(best):
        ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    print(f"[SAVED] {out_path}")
    plt.show()


def export_csv(rows, out_path):
    fieldnames = ["dataset", "variant", "method", "auprc", "fpr95", "images_used"]
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in sorted(rows, key=lambda x: (
            DATASET_ORDER.index(x["dataset"]) if x["dataset"] in DATASET_ORDER else 99,
            VARIANT_ORDER.index(x["variant"]) if x["variant"] in VARIANT_ORDER else 99,
            METHOD_ORDER.index(x["method"]) if x["method"] in METHOD_ORDER else 99,
        )):
            writer.writerow({k: r.get(k) for k in fieldnames})
    print(f"[SAVED] {out_path}")


## Generate figures and table

In [ ]:
export_csv(rows, out_dir / f"report_table_{TEMPERATURE}.csv")
save_heatmap(rows, "auprc", out_dir / f"auprc_heatmap_{TEMPERATURE}.png", f"AuPRC (%) at {TEMPERATURE}")
save_heatmap(rows, "fpr95", out_dir / f"fpr95_heatmap_{TEMPERATURE}.png", f"FPR95 (%) at {TEMPERATURE}")
save_scatter(rows, out_dir / f"auprc_vs_fpr95_{TEMPERATURE}.png")
save_best_bar(rows, out_dir / f"best_method_per_dataset_{TEMPERATURE}.png")
print("\nDone — all EDA outputs written to:", out_dir)
